# Adding Metadata | Cleaning the enriched dataset | Selecting the corpus

The following python code walks through the proces of creating a balanced dataset for stylometric anlysis. The names of the authors have already been standardized in OpenRefine. Metadata such a the *publication date* and *genre* of the novel,  as well as the *gender* and *country of citizenship* have been reconciled against Wikidata using OpenRefine. The Downloads column represents the number of downloads from the Project Gutenberg webiste, and will be the basis for the selection of novels, alongside country of citizenship.

The goal is to standardize genre and country of citizenship, and to make a balanced selection of novels

In [1]:
import pandas as pd

df = pd.read_csv("sf-fantasy-novels-csv-enriched.csv")

df.head(10)

,Title,genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,"United Kingdom of Great Britain and Ireland, K...",female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",1897-05-26T00:00:00Z,Bram Stoker,United Kingdom of Great Britain and Ireland,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",1900-01-01T00:00:00Z,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,"United Kingdom of Great Britain and Ireland, K...",female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...
5,Gulliver's Travels,"satire, fantasy, sea adventure novel",1726-10-28T00:00:00Z,Jonathan Swift,Kingdom of Ireland,male,17684,https://www.gutenberg.org/cache/epub/829/pg829...
6,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,"United Kingdom of Great Britain and Ireland, K...",female,14471,https://www.gutenberg.org/cache/epub/42324/pg4...
7,Carmilla,"fantasy, lesbian literature, vampire fiction, ...",1872-01-01T00:00:00Z,Sheridan Le Fanu,"United Kingdom of Great Britain and Ireland, B...",male,12860,https://www.gutenberg.org/cache/epub/10007/pg1...
8,Rip Van Winkle,time-travel fiction,1818-01-01T00:00:00Z,Washington Irving,United States,male,11659,https://www.gutenberg.org/cache/epub/60976/pg6...
9,Twenty Thousand Leagues Under the Sea,"science fiction, nautical fiction, scientific ...",1869-03-20T00:00:00Z,Jules Verne,France,male,10353,https://www.gutenberg.org/cache/epub/164/pg164...


### Standardizing country of citizenship

The United Kingdom has had multiple names over history:
- United Kingdom
- United Kingdom of Great Britain and Ireland
- United Kingdom of Great Britain
- British Empire

The reconciled values from wikidata have to be standardized into "United Kingdom" for the purpose of later analysis. The following function standardizes these values. This is not neccesary for the United States: these values are already consistant. We are collecting a corpus of US and UK authored novels, so we can ignore other nationalities.

In [2]:
def standardize_nationality(column):
    """
    Takes the DataFrame and standardizes UK values
    """
    uk_variants = ["United Kingdom","United Kingdom of Great Britain and Ireland","United Kingdom of Great Britain","British Empire"]
    
    for variant in uk_variants:
        if isinstance(column, str):  #error handling for null values
            if variant in column:
                return "United Kingdom"

    return column #keeps old values in case of no match

df["country of citizenship"] = df["country of citizenship"].apply(standardize_nationality)
df.head(10)

,Title,genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",1897-05-26T00:00:00Z,Bram Stoker,United Kingdom,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",1900-01-01T00:00:00Z,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...
5,Gulliver's Travels,"satire, fantasy, sea adventure novel",1726-10-28T00:00:00Z,Jonathan Swift,Kingdom of Ireland,male,17684,https://www.gutenberg.org/cache/epub/829/pg829...
6,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,14471,https://www.gutenberg.org/cache/epub/42324/pg4...
7,Carmilla,"fantasy, lesbian literature, vampire fiction, ...",1872-01-01T00:00:00Z,Sheridan Le Fanu,United Kingdom,male,12860,https://www.gutenberg.org/cache/epub/10007/pg1...
8,Rip Van Winkle,time-travel fiction,1818-01-01T00:00:00Z,Washington Irving,United States,male,11659,https://www.gutenberg.org/cache/epub/60976/pg6...
9,Twenty Thousand Leagues Under the Sea,"science fiction, nautical fiction, scientific ...",1869-03-20T00:00:00Z,Jules Verne,France,male,10353,https://www.gutenberg.org/cache/epub/164/pg164...


### Standardizing Genre
The data in this dataframe was scraped from the Science Fiction & Fantasy category available on Project Gutenberg. However, to classify the genres more strictly, the genre column in the dataframe contains reconciled Wikidata values of the novels. for the scope of this research, we want to strictly look at Fanatasy novels, Science Fiction novels or a combination of both. Therefore, we have to standardize the wikidata genres in the code below.

In [3]:
def categorize_genre(genre_column):
    """
    Takes a string (wikidata genres) and returns a standardized genre:
    'science fiction', 'fantasy', 'both', or '' if neither.
    """
    # Ensure the genre string is lowercase for consistent matching
    genre_lower = str(genre_column).lower()
    
    has_scifi = "science fiction" in genre_lower    #boolean
    has_fantasy = "fantasy" in genre_lower          #boolean

    if has_scifi and has_fantasy:
        return "fantasy, science fiction"
    elif has_scifi:
        return "science fiction"
    elif has_fantasy:
        return "fantasy"
    else:
        return None  # None is recognized by pandas .dropna() which will be used later

In [4]:
stand_genres = df['genre'].apply(categorize_genre)

df.insert(2, "standardized genre", stand_genres)

In [5]:
df.head(2)

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",None,1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...


### Filtering country of citizenship
Here we alter the dataframe to only include US and UK authors

In [6]:
uk_df = df[df["country of citizenship"] == "United Kingdom"]
us_df = df[df["country of citizenship"] == "United States"]

df = pd.concat([uk_df, us_df], ignore_index=True)
df = df.sort_values("Downloads", ascending=False).reset_index(drop=True)  #Here we sort again on based Downloads and reset the index of the df
df.head()

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",None,1895-01-01T00:00:00Z,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",None,1897-05-26T00:00:00Z,Bram Stoker,United Kingdom,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",fantasy,1900-01-01T00:00:00Z,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818-01-01T00:00:00Z,Mary Shelley,United Kingdom,female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...


### Filtering time period

In [7]:
#this codes changes the datatype of the publication date column to datetime
df['publication date'] = pd.to_datetime(df['publication date'], errors='coerce', utc=True)
df['publication date'].dtype

datetime64[ns, UTC]

The function below converts the datetime object to integers representing the publication year for standardization purposes for analysis in Stylo

In [8]:
import re

def extract_year(date_str):
    match = re.search(r'\b(18|19|20)\d{2}\b', str(date_str))
    return int(match.group()) if match else None

df['publication date'] = df['publication date'].apply(extract_year)
df['publication date'] = df['publication date'].astype("Int64") #convert float to int
df

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818,Mary Shelley,United Kingdom,female,264413,https://www.gutenberg.org/cache/epub/84/pg84.txt
1,The King in Yellow,"Decadent movement, horror literature, supernat...",None,1895,Robert W. Chambers,United States,male,68842,https://www.gutenberg.org/cache/epub/8492/pg84...
2,Dracula,"epistolary novel, invasion literature, Gothic ...",None,1897,Bram Stoker,United Kingdom,male,42071,https://www.gutenberg.org/cache/epub/345/pg345...
3,The Wonderful Wizard of Oz,"children's story, fantasy",fantasy,1900,L. Frank Baum,United States,male,24328,https://www.gutenberg.org/cache/epub/55/pg55.txt
4,"Frankenstein; or, The Modern Prometheus","epistolary novel, body horror literature, Goth...","fantasy, science fiction",1818,Mary Shelley,United Kingdom,female,20274,https://www.gutenberg.org/cache/epub/41445/pg4...
...,...,...,...,...,...,...,...,...,...
2980,Nahkaratti (Finnish),NaN,None,<NA>,Arthur Conan Doyle,United Kingdom,male,166,https://www.gutenberg.org/cache/epub/56573/pg5...
2981,Cognitive Behavioral Therapy Treatment for a W...,NaN,None,2019,William Campbell Gault,United States,male,165,https://www.gutenberg.org/cache/epub/74139/pg7...
2982,Punainen rutto (Finnish),NaN,None,<NA>,Jack London,United States,male,164,https://www.gutenberg.org/cache/epub/56785/pg5...
2983,The Reluctant Weapon,NaN,None,<NA>,Howard L. Myers,United States,male,164,https://www.gutenberg.org/cache/epub/31995/pg3...


In [9]:
# #Create a seelction of 19th century novels
df = df[(df['publication date'] >= 1880) & (df['publication date'] < 1980)]

#filter out all novels that are not classified as fiction, science fiction or both
df = df.dropna(subset=['standardized genre']) 

#check for dupicate novels titles (this happens sometimes on Project GUtenberg)
# df.value_counts(subset=['Title','Author']).head()

#### As we can see, there are some duplicates which we should remove.

In [10]:
df.shape

(330, 9)

In [11]:
df = df.drop_duplicates(subset=['Title','Author'])  
df.value_counts(subset=['Title','Author'])

Title                             Author              
"Let There Be Light"              H. B. Fyfe              1
The Island of Dr Moreau           H. G. Wells             1
The Man Who Saved the Earth       Austin Hall             1
The Magical Mimics in Oz          Jack Snow               1
The Magic of Oz                   L. Frank Baum           1
                                                         ..
Pattern for Conquest              George O. Smith         1
Palos of the Dog Star Pack        John Ulrich Giesy       1
Ozoplaning with the Wizard of Oz  Ruth Plumly Thompson    1
Ozma of Oz                        L. Frank Baum           1
Worlds of the Imperium            Keith Laumer            1
Name: count, Length: 294, dtype: int64

#### To prevent high similarity between novels by the same author we limit the dataset to 1 novel per author. This way, we isolate genre en country of citizenship as factors for stylistic similarity. The first novel for eacht other in the df (with the highest amount of downloads) is kept.


In [12]:
df = df.drop_duplicates(subset=['Author'])

#### In the code below we see how many novels are left per category after all the previous filtering of the corpus.

In [13]:
df.value_counts(['country of citizenship','standardized genre'])

country of citizenship  standardized genre      
United States           science fiction             66
United Kingdom          fantasy                     16
United States           fantasy                     15
United Kingdom          science fiction             12
United States           fantasy, science fiction     6
Name: count, dtype: int64

#### As we can see, the category with the lowest amount of novels is Science Fiction by UK authors with 12 novels. After manual evaluation, we found that one novel in the UK Scifi category was wrongly attributed to Scifi (On the Borderland by Frederick Britten Austin), which brings us to 11. We will limit the other categories to 11 novels as well, so that the dataset is balanced. (novels classified as both sci-fi and fantasy will be left out, since they cannot be categorized clearly)

In [14]:
#US Sci-Fi dataframe (selects the first 11 novels with the highest downloads)
us_scifi = df[(df['country of citizenship'] == 'United States') & (df['standardized genre'] == 'science fiction')].head(11).reset_index(drop=True)

#UK sci-fi dataframe (selects the first 11 with the highest downloads)
uk_scifi = df[(df['country of citizenship'] == 'United Kingdom') & (df['standardized genre'] == 'science fiction')].head(12).reset_index(drop=True)

#US Fantasy dataframe (selects the first 11 with the highest downloads)
us_fantasy = df[(df['country of citizenship'] == 'United States') & (df['standardized genre'] == 'fantasy')].head(11).reset_index(drop=True)

#UK Fantasy dataframe (selects the first 11 with the highest downloads)
uk_fantasy = df[(df['country of citizenship'] == 'United Kingdom') & (df['standardized genre'] == 'fantasy')].head(11).reset_index(drop=True)

# Corpus Curation

Since we did automatic reconciliation in OpenRefine, it is essential to check whether the Titles & Authors match the actual .txt files.

The links you see below have been manually checked.

### US Fantasy

In [15]:
for title, author, link in zip(us_fantasy['Title'], us_fantasy['Author'], us_fantasy['Plain Text UTF-8 Link']):
    print(title," ", author, link)

The Wonderful Wizard of Oz   L. Frank Baum https://www.gutenberg.org/cache/epub/55/pg55.txt
Red Nails   Robert E. Howard https://www.gutenberg.org/cache/epub/32759/pg32759.txt
David and the Phoenix   Edward Ormondroyd https://www.gutenberg.org/cache/epub/27922/pg27922.txt
Jack Pumpkinhead of Oz   Ruth Plumly Thompson https://www.gutenberg.org/cache/epub/75720/pg75720.txt
The Blue Star   Fletcher Pratt https://www.gutenberg.org/cache/epub/56889/pg56889.txt
Jurgen, A Comedy of Justice   James Branch Cabell https://www.gutenberg.org/cache/epub/8771/pg8771.txt
Davy and the Goblin   Charles E. Carryl https://www.gutenberg.org/cache/epub/25031/pg25031.txt
Mr. Munchausen   John Kendrick Bangs https://www.gutenberg.org/cache/epub/33432/pg33432.txt
The Return of Tharn   Howard Browne https://www.gutenberg.org/cache/epub/33529/pg33529.txt
The Magical Mimics in Oz   Jack Snow https://www.gutenberg.org/cache/epub/56555/pg56555.txt
Khaled: A Tale of Arabia   Francis Marion Crawford https://www.gute

This subcorpus is correct

### UK Fantasy

In [16]:
for title, author, link in zip(uk_fantasy['Title'], uk_fantasy['Author'], uk_fantasy['Plain Text UTF-8 Link']):
    print(title," ", author, link)

The House of the Wolfings   William Morris https://www.gutenberg.org/cache/epub/2885/pg2885.txt
Lilith   George MacDonald https://www.gutenberg.org/cache/epub/1640/pg1640.txt
The House on the Borderland   William Hope Hodgson https://www.gutenberg.org/cache/epub/10002/pg10002.txt
Five Children and It   Edith Nesbit https://www.gutenberg.org/cache/epub/17314/pg17314.txt
She: A History of Adventure   H. Rider Haggard https://www.gutenberg.org/cache/epub/3155/pg3155.txt
The Eternal Moment   E. M. Forster https://www.gutenberg.org/cache/epub/72890/pg72890.txt
The King of Elfland's Daughter   Edward Plunkett, 18th Baron of Dunsany https://www.gutenberg.org/cache/epub/61077/pg61077.txt
Peter Pan in Kensington Gardens   J. M. Barrie https://www.gutenberg.org/cache/epub/26998/pg26998.txt
Sylvie and Bruno   Lewis Carroll https://www.gutenberg.org/cache/epub/620/pg620.txt
Puck of Pook's Hill   Rudyard Kipling https://www.gutenberg.org/cache/epub/26027/pg26027.txt
The Lair of the White Worm   Bra

This subcorpus is correct

### US Sci-Fi

In [17]:
for title, author, link in zip(us_scifi['Title'], us_scifi['Author'], us_scifi['Plain Text UTF-8 Link']):
    print(title," ", author, link)

The Game of Rat and Dragon   Cordwainer Smith https://www.gutenberg.org/cache/epub/29614/pg29614.txt
The Call of Cthulhu   H. P. Lovecraft https://www.gutenberg.org/cache/epub/68283/pg68283.txt
2 B R 0 2 B   Kurt Vonnegut https://www.gutenberg.org/cache/epub/21279/pg21279.txt
Herland   Charlotte Perkins Gilman https://www.gutenberg.org/cache/epub/32/pg32.txt
Second Variety   Philip K. Dick https://www.gutenberg.org/cache/epub/32032/pg32032.txt
The Iron Heel   Jack London https://www.gutenberg.org/cache/epub/1164/pg1164.txt
A Martian Odyssey   Stanley G. Weinbaum https://www.gutenberg.org/cache/epub/23731/pg23731.txt
The Marching Morons   Cyril M. Kornbluth https://www.gutenberg.org/cache/epub/51233/pg51233.txt
The Colors of Space   Marion Zimmer Bradley https://www.gutenberg.org/cache/epub/20796/pg20796.txt
Etidorhpa   John Uri Lloyd https://www.gutenberg.org/cache/epub/37775/pg37775.txt
Spacehounds of IPC   Edward Elmer Smith https://www.gutenberg.org/cache/epub/20857/pg20857.txt


This selection is correct!

### UK Sci-Fi

In [18]:
for title, author, link in zip(uk_scifi['Title'], uk_scifi['Author'], uk_scifi['Plain Text UTF-8 Link']):
    print(title," ", author, link)

The Time Machine   H. G. Wells https://www.gutenberg.org/cache/epub/35/pg35.txt
Flatland   Edwin Abbott Abbott https://www.gutenberg.org/cache/epub/201/pg201.txt
The Poison Belt   Arthur Conan Doyle https://www.gutenberg.org/cache/epub/126/pg126.txt
Lord of the World   Robert Hugh Benson https://www.gutenberg.org/cache/epub/14021/pg14021.txt
The Purple Cloud   M. P. Shiel https://www.gutenberg.org/cache/epub/11229/pg11229.txt
The World Below   S. Fowler Wright https://www.gutenberg.org/cache/epub/76352/pg76352.txt
Across the Zodiac   Percy Greg https://www.gutenberg.org/cache/epub/10165/pg10165.txt
Of All Possible Worlds   William Tenn https://www.gutenberg.org/cache/epub/50948/pg50948.txt
The Lost Kafoozalum   Pauline Ashwell https://www.gutenberg.org/cache/epub/30427/pg30427.txt
The Borderland of Sol   Frederick Britten Austin https://www.gutenberg.org/cache/epub/65837/pg65837.txt
New Amazonia   Elizabeth Burgoyne Corbett https://www.gutenberg.org/cache/epub/56876/pg56876.txt
The Ham

'The Borderland of Sol' was wrongly attributed to 'On the Borderland' by F.B. Austin, in this novel does not qualify as strictly sci-fi.

In [19]:
uk_scifi = uk_scifi.drop([9]).reset_index(drop=True)
uk_scifi

,Title,genre,standardized genre,publication date,Author,country of citizenship,sex or gender,Downloads,Plain Text UTF-8 Link
0,The Time Machine,"time-travel fiction, romantic fiction, dystopi...",science fiction,1895,H. G. Wells,United Kingdom,male,10268,https://www.gutenberg.org/cache/epub/35/pg35.txt
1,Flatland,"science fiction, mathematical fiction, satiric...",science fiction,1884,Edwin Abbott Abbott,United Kingdom,male,2577,https://www.gutenberg.org/cache/epub/201/pg201...
2,The Poison Belt,science fiction,science fiction,1913,Arthur Conan Doyle,United Kingdom,male,1040,https://www.gutenberg.org/cache/epub/126/pg126...
3,Lord of the World,"dystopian fiction, apocalyptic fiction, scienc...",science fiction,1908,Robert Hugh Benson,United Kingdom,male,786,https://www.gutenberg.org/cache/epub/14021/pg1...
4,The Purple Cloud,"post-apocalyptic fiction, serialized fiction, ...",science fiction,1901,M. P. Shiel,United Kingdom,male,742,https://www.gutenberg.org/cache/epub/11229/pg1...
5,The World Below,science fiction,science fiction,1929,S. Fowler Wright,United Kingdom,male,528,https://www.gutenberg.org/cache/epub/76352/pg7...
6,Across the Zodiac,science fiction,science fiction,1880,Percy Greg,United Kingdom,male,516,https://www.gutenberg.org/cache/epub/10165/pg1...
7,Of All Possible Worlds,science fiction,science fiction,1955,William Tenn,United Kingdom,male,369,https://www.gutenberg.org/cache/epub/50948/pg5...
8,The Lost Kafoozalum,science fiction,science fiction,1960,Pauline Ashwell,United Kingdom,female,364,https://www.gutenberg.org/cache/epub/30427/pg3...
9,New Amazonia,"utopian literature, feminist science fiction, ...",science fiction,1889,Elizabeth Burgoyne Corbett,United Kingdom,female,297,https://www.gutenberg.org/cache/epub/56876/pg5...


### Now that we have 4 dataframes representing the four categories, we can scrape the txt files from Project Gutenberg

In [20]:
import requests
import os

#UK fantasy txt scraping (this will overwrite any existing files in the repo!)
for idx, row in uk_fantasy.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/uk_fantasy", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)

In [21]:
#UK scifi txt scraping
for idx, row in uk_scifi.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/uk_scifi", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)

In [22]:
#US scifi txt scraping
for idx, row in us_scifi.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/us_scifi", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)

In [23]:
#US fantasy txt scraping
for idx, row in us_fantasy.iterrows():   
    author = row['Author']
    url = row['Plain Text UTF-8 Link']

    response = requests.get(url)

    filename = f"{author}.txt"
    filepath = os.path.join("data/us_fantasy", filename)
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)